# U-Mamba · SAR + optical time series → plantation locations, crop type, yield
**Notebook 1 of 2 in the U-Mamba vs SITSMamba comparison** (plus an optional pretrained U-TAE baseline and a comparison notebook).

| Stage | Dataset | Inputs | Predicts |
|---|---|---|---|
| 1 | **PASTIS-R** (France, 2,433 patches) | Sentinel-1 SAR (asc + desc) + Sentinel-2 time series | crop type (18 classes), field extent, boundaries → individual fields |
| 2 | **YieldSAT** (4 countries, 2,173 fields) | Sentinel-2 + weather + soil + terrain | **yield (t/ha)**, crop type, field extent |

Stage 2 starts from the stage-1 weights (band-by-band input transfer, new static and yield branches).
Run the cells from top to bottom. Everything is saved to `MyDrive/sar_crop_ml/`, and re-running after a disconnect resumes where training stopped.
Recommended runtime: **A100 / H100 / L4 GPU, High-RAM**.

## 0 · Setup
Mounts Google Drive, creates the shared folder layout and installs the packages.

**All notebooks use the same Drive layout** (`MyDrive/sar_crop_ml/`):

| Folder | Contents |
|---|---|
| `data/raw/` | downloaded archives: `PASTIS-R.zip`; YieldSAT files go in `data/raw/yieldsat/` |
| `data/cache/` | preprocessed tensors (reused by every notebook) |
| `data/splits/` | fold assignments (identical for every model) |
| `weights/pretrained/` | third-party weights (U-TAE) |
| `runs/<model>/<dataset>/<run>/` | `config.json`, `checkpoints/last.pt` + `best.pt`, `metrics.csv`, `log.txt`, `test_metrics.json`, sample predictions, environment info |

A run that stops (time-out, disconnect, or a `STOP` file you create in its folder) resumes from `last.pt` when you re-run the same cell, including in the middle of an epoch.

In [ ]:
import os, sys, json, time, math, shutil, subprocess, glob, random, hashlib, platform, re, zipfile, datetime as dt
from pathlib import Path

SMOKE = os.environ.get('SARCROP_SMOKE') == '1'          # tiny synthetic run used to test these notebooks
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not SMOKE:
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_ROOT = Path(os.environ.get('SARCROP_ROOT', '/content/drive/MyDrive/sar_crop_ml' if IN_COLAB else './sar_crop_ml')).resolve()
LOCAL_ROOT = Path(os.environ.get('SARCROP_LOCAL', '/content/sar_crop_local' if IN_COLAB else './sar_crop_local')).resolve()
P = dict(
    raw=DRIVE_ROOT / 'data/raw', cache=DRIVE_ROOT / 'data/cache', splits=DRIVE_ROOT / 'data/splits',
    weights=DRIVE_ROOT / 'weights', runs=DRIVE_ROOT / 'runs', reports=DRIVE_ROOT / 'reports',
    local=LOCAL_ROOT,  # fast local SSD: extracted data and training cache live here during a session
)
for p in P.values():
    p.mkdir(parents=True, exist_ok=True)
print('Drive root :', DRIVE_ROOT)
print('Local root :', LOCAL_ROOT)

def sh(cmd, check=True, timeout=None):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-3000:])
    if r.returncode:
        print(f'[exit code {r.returncode}] stderr (last 4000 chars):\n{r.stderr[-4000:]}')
        if check: raise RuntimeError(r.stderr[-3000:])
    return r

if not SMOKE:
    sh('pip -q install xarray netCDF4 h5netcdf scikit-image einops pandas matplotlib', check=False)

In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F, torch.utils.checkpoint
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu'
BF16 = DEVICE == 'cuda' and torch.cuda.is_bf16_supported()
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == 'cuda' else 0
print(f'torch {torch.__version__} · device {GPU} · {GPU_MEM_GB:.0f} GB · bf16={BF16}')
torch.backends.cudnn.benchmark = True
if DEVICE == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

## 1 · Mamba layer
Uses the official CUDA kernels (`mamba_ssm`) when they are installed. Otherwise a pure-PyTorch implementation with the **same parameter names** is used, so checkpoints work with either. The fallback uses a log-space parallel scan: correct, but slower and more memory-hungry. Install the kernels on GPU runtimes; it takes 2–30 minutes depending on whether a prebuilt wheel matches Colab's PyTorch.

In [ ]:
INSTALL_MAMBA_KERNELS = not SMOKE and DEVICE == 'cuda'   # set False to force the PyTorch fallback
MambaCUDA = None
try:
    from mamba_ssm import Mamba as MambaCUDA
except Exception:
    if INSTALL_MAMBA_KERNELS:
        try:
            sh('pip install -q "causal-conv1d>=1.4.0" --no-build-isolation', check=False, timeout=2400)
            sh('pip install -q mamba-ssm --no-build-isolation', check=False, timeout=2400)
            from mamba_ssm import Mamba as MambaCUDA
        except Exception as e:
            print('mamba_ssm unavailable, using the PyTorch fallback:', str(e)[:200])
print('Mamba backend:', 'mamba_ssm CUDA kernels' if MambaCUDA else 'PyTorch fallback')


def _scan(u, delta, A, B, C, D):
    """Selective scan h_t = exp(dt*A) h_{t-1} + dt*B_t*u_t, y_t = C_t·h_t + D*u_t.
    Log-space parallel scan (Heinsen 2023) with positive/negative parts kept apart.
    u, delta: (b, l, d) · A: (d, n) · B, C: (b, l, n)."""
    u, delta, B, C = u.float(), delta.float(), B.float(), C.float()
    log_a = delta.unsqueeze(-1) * A                                   # (b,l,d,n), <= 0
    a_star = torch.cumsum(log_a, dim=1)
    bx = (delta * u).unsqueeze(-1) * B.unsqueeze(2)                   # (b,l,d,n)
    tiny = torch.finfo(torch.float32).tiny
    lp = torch.log(bx.clamp(min=0) + tiny) - a_star
    ln = torch.log((-bx).clamp(min=0) + tiny) - a_star
    h = torch.exp(a_star + torch.logcumsumexp(lp, dim=1)) - torch.exp(a_star + torch.logcumsumexp(ln, dim=1))
    y = (h * C.unsqueeze(2)).sum(-1) + u * D
    return y


def _scan_seq(u, delta, A, B, C, D):
    """Same recurrence, step by step: never materialises (b, l, d, n). Faster for short sequences (L <= 64)."""
    u, delta, B, C = u.float(), delta.float(), B.float(), C.float()
    b, l, d = u.shape
    h = u.new_zeros(b, d, A.shape[1])
    ys = []
    for t in range(l):
        h = torch.exp(delta[:, t, :, None] * A) * h + (delta[:, t] * u[:, t])[..., None] * B[:, t, None, :]
        ys.append(torch.einsum('bdn,bn->bd', h, C[:, t]))
    return torch.stack(ys, 1) + u * D


class MambaTorch(nn.Module):
    """Pure-PyTorch Mamba (v1) block, parameter-compatible with mamba_ssm.Mamba."""
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2, dt_rank='auto', dt_min=0.001, dt_max=0.1,
                 dt_init_floor=1e-4, conv_bias=True, bias=False, chunk=2048):
        super().__init__()
        self.d_inner = int(expand * d_model)
        self.d_state = d_state
        self.dt_rank = math.ceil(d_model / 16) if dt_rank == 'auto' else dt_rank
        self.chunk = chunk
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=bias)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, d_conv, groups=self.d_inner, padding=d_conv - 1, bias=conv_bias)
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)
        dt_init_std = self.dt_rank ** -0.5
        nn.init.uniform_(self.dt_proj.weight, -dt_init_std, dt_init_std)
        dt_ = torch.exp(torch.rand(self.d_inner) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)).clamp(min=dt_init_floor)
        with torch.no_grad():
            self.dt_proj.bias.copy_(dt_ + torch.log(-torch.expm1(-dt_)))
        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=bias)

    def forward(self, x):                      # (b, l, d_model)
        b, l, _ = x.shape
        xz = self.in_proj(x)
        u, z = xz.chunk(2, dim=-1)
        u = F.silu(self.conv1d(u.transpose(1, 2))[..., :l].transpose(1, 2))
        dbc = self.x_proj(u)
        dtr, Bm, Cm = dbc.split([self.dt_rank, self.d_state, self.d_state], dim=-1)
        delta = F.softplus(self.dt_proj(dtr))
        A = -torch.exp(self.A_log.float())
        # chunk the batch so the (b, l, d, n) scan tensors stay within memory
        scan = _scan_seq if l <= 64 else _scan
        budget = 4e8 if scan is _scan_seq else 6.4e7          # sequential scan keeps only (b, d, n) per step
        per = max(1, int(budget // ((1 if scan is _scan_seq else l) * self.d_inner * self.d_state)))
        # gradient checkpointing per chunk: the states are recomputed in backward, as the CUDA kernel does
        run = (lambda *a: torch.utils.checkpoint.checkpoint(scan, *a, use_reentrant=False)) if (self.training and torch.is_grad_enabled()) else scan
        ys = [run(u[i:i + per], delta[i:i + per], A, Bm[i:i + per], Cm[i:i + per], self.D.float()) for i in range(0, b, per)]
        y = torch.cat(ys, 0).to(x.dtype) * F.silu(z)
        return self.out_proj(y)


def make_mamba(d_model, d_state=16, d_conv=4, expand=2):
    if MambaCUDA is not None and DEVICE == 'cuda':
        return MambaCUDA(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
    return MambaTorch(d_model, d_state=d_state, d_conv=d_conv, expand=expand)


def _check_scan():
    """The parallel scan must match a plain sequential recurrence."""
    torch.manual_seed(0)
    b, l, d, n = 2, 37, 8, 4
    u, delta = torch.randn(b, l, d), torch.rand(b, l, d) * 0.5
    A, Bm, Cm, D = -torch.rand(d, n) * 2, torch.randn(b, l, n), torch.randn(b, l, n), torch.randn(d)
    h = torch.zeros(b, d, n); ys = []
    for t in range(l):
        h = torch.exp(delta[:, t, :, None] * A) * h + (delta[:, t] * u[:, t])[..., None] * Bm[:, t, None, :]
        ys.append((h * Cm[:, t, None, :]).sum(-1) + u[:, t] * D)
    ref = torch.stack(ys, 1)
    err = max((ref - _scan(u, delta, A, Bm, Cm, D)).abs().max().item(), (ref - _scan_seq(u, delta, A, Bm, Cm, D)).abs().max().item())
    assert err < 1e-3, f'scan mismatch {err}'
    print(f'selective scan check passed (max error {err:.1e})')
_check_scan()

## 2 · Dataset 1: PASTIS-R (Sentinel-1 SAR + Sentinel-2, France)
[PASTIS-R](https://github.com/VSainteuf/pastis-benchmark) has 2,433 patches of 128×128 px (10 m) over one season (Sep 2018 – Nov 2019):
Sentinel-2 (10 bands, 38–61 dates), Sentinel-1 ascending and descending (VV, VH, VV/VH; ~70 dates each), semantic crop labels (18 crops + background + void) and parcel instances. There are five official folds.

**Targets used here**
* `sem`: crop type per pixel (20 classes, void ignored)
* `extent`, `boundary`, `dist`: where the plantations / fields are (parcel mask, parcel edges, distance to the edge). Watershed on these gives individual fields.

**Time handling**: every series is put on one fixed grid (default 24 steps of 18 days). Each step stores the mean of the observations falling in it (median when there are 3 or more), or the nearest observation plus an `OBS_*` flag of 0 when the step is empty. The same grid length is used for YieldSAT, which comes with 24 steps.

In [ ]:
PASTIS_CLASSES = ['Background', 'Meadow', 'Soft winter wheat', 'Corn', 'Winter barley', 'Winter rapeseed', 'Spring barley',
                  'Sunflower', 'Grapevine', 'Beet', 'Winter triticale', 'Winter durum wheat', 'Fruits, vegetables, flowers',
                  'Potatoes', 'Leguminous fodder', 'Soybeans', 'Orchard', 'Mixed cereal', 'Sorghum', 'Void label']
PASTIS_VOID = 19
S2_PASTIS = ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']
S1_NAMES = ['S1A_VV', 'S1A_VH', 'S1A_RATIO', 'S1D_VV', 'S1D_VH', 'S1D_RATIO']
PASTIS_DYN = S2_PASTIS + S1_NAMES + ['OBS_S2', 'OBS_S1']
PASTIS_URL = 'https://zenodo.org/api/records/5735646/files/PASTIS-R.zip/content'   # 53.7 GB
PASTIS_GRID = dict(start='2018-09-01', steps=24, step_days=18)
KEEP_RAW_ZIP_ON_DRIVE = True        # keeps the 54 GB zip in Drive so later sessions skip the download
CACHE_PASTIS_ON_DRIVE = False       # also keep the preprocessed tensors (~34 GB) on Drive (faster restarts, uses space)


def get_pastis_root():
    """Extract PASTIS-R to the local SSD, downloading the zip only once (kept on Drive)."""
    root = P['local'] / 'PASTIS-R'
    if (root / 'metadata.geojson').exists():
        return root
    if SMOKE:
        make_fake_pastis(root); return root
    drive_zip = P['raw'] / 'PASTIS-R.zip'
    local_zip = P['local'] / 'PASTIS-R.zip'
    src = drive_zip if drive_zip.exists() and drive_zip.stat().st_size > 5e10 else None
    if src is None:
        sh(f'wget -c -q --show-progress -O "{local_zip}" "{PASTIS_URL}"', timeout=6 * 3600)
        src = local_zip
        if KEEP_RAW_ZIP_ON_DRIVE:
            sh(f'cp "{local_zip}" "{drive_zip}.part" && mv "{drive_zip}.part" "{drive_zip}"', check=False)
    print('extracting', src)
    sh(f'unzip -q -o "{src}" -d "{P["local"]}"', timeout=3 * 3600)
    if local_zip.exists():
        local_zip.unlink()          # free local disk; the Drive copy (if kept) is the cache
    cand = [p.parent for p in P['local'].rglob('metadata.geojson')]
    if cand and cand[0] != root:
        cand[0].rename(root)
    return root


def make_fake_pastis(root, n=12, H=32):
    """Tiny synthetic PASTIS-R with the same file layout (smoke tests only)."""
    rng = np.random.default_rng(0)
    for d in ['DATA_S2', 'DATA_S1A', 'DATA_S1D', 'ANNOTATIONS', 'INSTANCE_ANNOTATIONS']:
        (root / d).mkdir(parents=True, exist_ok=True)
    feats = []
    for i in range(n):
        pid = 10000 + i
        def dates(k):
            ds = sorted(rng.choice(np.arange(0, 420), k, replace=False))
            return {str(j): int((dt.date(2018, 9, 1) + dt.timedelta(days=int(x))).strftime('%Y%m%d')) for j, x in enumerate(ds)}
        d2, da, dd = dates(int(rng.integers(10, 16))), dates(int(rng.integers(12, 18))), dates(int(rng.integers(12, 18)))
        inst = np.zeros((H, H), np.int32); sem = np.zeros((H, H), np.int32)
        k = 1
        for y0 in range(0, H, 8):
            for x0 in range(0, H, 8):
                if rng.random() < 0.8:
                    inst[y0:y0 + 7, x0:x0 + 7] = k; sem[y0:y0 + 7, x0:x0 + 7] = rng.integers(1, 20); k += 1
        base = sem[None, None].astype(np.float32)
        np.save(root / f'DATA_S2/S2_{pid}.npy', (rng.normal(1000, 200, (len(d2), 10, H, H)) + 30 * base).astype(np.int16))
        np.save(root / f'DATA_S1A/S1A_{pid}.npy', (rng.normal(-12, 2, (len(da), 3, H, H)) + 0.2 * base).astype(np.float32))
        np.save(root / f'DATA_S1D/S1D_{pid}.npy', (rng.normal(-12, 2, (len(dd), 3, H, H)) + 0.2 * base).astype(np.float32))
        np.save(root / f'ANNOTATIONS/TARGET_{pid}.npy', np.stack([sem, sem, sem]))
        np.save(root / f'INSTANCE_ANNOTATIONS/INSTANCES_{pid}.npy', inst)
        feats.append({'type': 'Feature', 'geometry': None, 'properties': {
            'ID_PATCH': pid, 'Fold': i % 5 + 1, 'dates-S2': json.dumps(d2), 'dates-S1A': json.dumps(da), 'dates-S1D': json.dumps(dd)}})
    (root / 'metadata.geojson').write_text(json.dumps({'type': 'FeatureCollection', 'features': feats}))


def _read_meta(root):
    feats = json.loads((root / 'metadata.geojson').read_text())['features']
    meta = {}
    for f in feats:
        p = f['properties']
        d = {}
        for s in ('S2', 'S1A', 'S1D'):
            v = p.get(f'dates-{s}')
            v = json.loads(v) if isinstance(v, str) else (v or {})
            d[s] = np.array([(dt.date(int(str(x)[:4]), int(str(x)[4:6]), int(str(x)[6:8])) - dt.date.fromisoformat(PASTIS_GRID['start'])).days
                             for _, x in sorted(v.items(), key=lambda kv: int(kv[0]))])
        meta[int(p['ID_PATCH'])] = dict(fold=int(p['Fold']), dates=d)
    return meta


def _norm_stats(root, meta, sat, nch):
    f = root / f'NORM_{sat}_patch.json'
    if f.exists():
        js = json.loads(f.read_text())
        m = np.mean([js[k]['mean'] for k in js], 0); s = np.mean([js[k]['std'] for k in js], 0)
        return np.array(m, np.float32), np.array(s, np.float32)
    ids = list(meta)[:: max(1, len(meta) // 80)]
    xs = [np.load(root / f'DATA_{sat}/{sat}_{i}.npy', mmap_mode='r')[:, :, ::8, ::8].astype(np.float32) for i in ids]
    x = np.concatenate([a.transpose(1, 0, 2, 3).reshape(nch, -1) for a in xs], 1)
    return np.nanmean(x, 1).astype(np.float32), (np.nanstd(x, 1) + 1e-6).astype(np.float32)


def _to_grid(arr, days, grid):
    half = (grid[1] - grid[0]) / 2 if len(grid) > 1 else 1e9
    out = np.zeros((len(grid),) + arr.shape[1:], np.float32); obs = np.zeros(len(grid), np.float32)
    for g, gd in enumerate(grid):
        sel = np.abs(days - gd) <= half
        if sel.sum() >= 3: out[g] = np.median(arr[sel], 0); obs[g] = 1
        elif sel.any(): out[g] = arr[sel].mean(0); obs[g] = 1
        else: out[g] = arr[int(np.argmin(np.abs(days - gd)))]
    return out, obs


def parcel_targets(inst, sem):
    """extent (field vs background), boundary (parcel edges) and distance-to-edge (0..1 per parcel)."""
    from scipy import ndimage
    extent = (sem > 0).astype(np.uint8)
    pad = np.pad(inst, 1, mode='edge')
    edge = np.zeros_like(inst, bool)
    for dy, dx in ((0, 1), (1, 0), (2, 1), (1, 2)):
        edge |= pad[dy:dy + inst.shape[0], dx:dx + inst.shape[1]] != inst
    boundary = (edge & (inst > 0)).astype(np.uint8)
    dist = np.zeros(inst.shape, np.float32)
    for k in np.unique(inst):
        if k == 0: continue
        m = inst == k
        d = ndimage.distance_transform_edt(np.pad(m, 1))[1:-1, 1:-1]
        dist[m] = d[m] / max(d.max(), 1)
    return extent, boundary, dist


def _prep_one(args):
    root, out, pid, info, grid, norms = args
    f = out / f'{pid}.npz'
    if f.exists(): return pid
    parts, obs = [], {}
    for sat, nch in (('S2', 10), ('S1A', 3), ('S1D', 3)):
        a = np.load(root / f'DATA_{sat}/{sat}_{pid}.npy').astype(np.float32)
        a = (a - norms[sat][0][None, :, None, None]) / norms[sat][1][None, :, None, None]
        a = np.nan_to_num(a)
        g, o = _to_grid(a, info['dates'][sat], grid)
        parts.append(g); obs[sat] = o
    T, _, H, W = parts[0].shape
    flags = np.stack([np.broadcast_to(obs['S2'][:, None, None], (T, H, W)),
                      np.broadcast_to(np.maximum(obs['S1A'], obs['S1D'])[:, None, None], (T, H, W))], 1)
    x = np.concatenate(parts + [flags], 1).astype(np.float16)
    sem = np.load(root / f'ANNOTATIONS/TARGET_{pid}.npy')[0].astype(np.int64)
    inst = np.load(root / f'INSTANCE_ANNOTATIONS/INSTANCES_{pid}.npy').astype(np.int32)
    extent, boundary, dist = parcel_targets(inst, sem)
    sem_t = np.where(sem == PASTIS_VOID, 255, sem).astype(np.uint8)
    np.savez(f.with_suffix('.tmp.npz'), x=x, days=np.array(grid, np.int16), sem=sem_t, extent=extent, boundary=boundary,
             dist=dist.astype(np.float16), inst=inst)
    os.replace(f.with_suffix('.tmp.npz'), f)
    return pid


def prepare_pastis():
    """Returns (cache_dir, meta). Preprocessing is resumable (existing patches are skipped)."""
    tag = f"pastis_T{PASTIS_GRID['steps']}x{PASTIS_GRID['step_days']}"
    out = P['local'] / 'cache' / tag
    drive_out = P['cache'] / tag
    out.mkdir(parents=True, exist_ok=True)
    if CACHE_PASTIS_ON_DRIVE and (drive_out / 'DONE').exists() and not (out / 'DONE').exists():
        sh(f'cp -r "{drive_out}/." "{out}/"')
    root = get_pastis_root()
    meta = _read_meta(root)
    if not (out / 'DONE').exists():
        norms = {s: _norm_stats(root, meta, s, n) for s, n in (('S2', 10), ('S1A', 3), ('S1D', 3))}
        json.dump({s: [v[0].tolist(), v[1].tolist()] for s, v in norms.items()}, open(out / 'norm.json', 'w'))
        grid = [i * PASTIS_GRID['step_days'] + PASTIS_GRID['step_days'] / 2 for i in range(PASTIS_GRID['steps'])]
        jobs = [(root, out, pid, info, grid, norms) for pid, info in meta.items()]
        from concurrent.futures import ProcessPoolExecutor
        t0 = time.time()
        with ProcessPoolExecutor(max(1, min(16, os.cpu_count() or 2))) as ex:
            for k, _ in enumerate(ex.map(_prep_one, jobs, chunksize=4)):
                if k % 200 == 0: print(f'  preprocessed {k}/{len(jobs)} patches · {time.time() - t0:.0f}s', flush=True)
        (out / 'DONE').write_text(str(len(jobs)))
        if CACHE_PASTIS_ON_DRIVE:
            drive_out.mkdir(parents=True, exist_ok=True)
            sh(f'cp -r "{out}/." "{drive_out}/"', check=False)
    # official fold assignment, saved once and shared by every notebook
    sp = P['splits'] / 'pastis_folds.json'
    if not sp.exists():
        json.dump({str(k): v['fold'] for k, v in meta.items()}, open(sp, 'w'))
    folds = {int(k): v for k, v in json.load(open(sp)).items()}
    return out, folds


def pastis_fold_split(folds, test_fold):
    val_fold = test_fold % 5 + 1
    ids = sorted(folds)
    return ([i for i in ids if folds[i] not in (test_fold, val_fold)],
            [i for i in ids if folds[i] == val_fold],
            [i for i in ids if folds[i] == test_fold])


def augment_spatial(arrs, rng):
    k = int(rng.integers(4)); flip = rng.random() < 0.5
    out = []
    for a in arrs:
        if a is None: out.append(None); continue
        a = np.rot90(a, k, axes=(-2, -1))
        if flip: a = a[..., ::-1]
        out.append(np.ascontiguousarray(a))
    return out


class PastisDS(torch.utils.data.Dataset):
    def __init__(self, cache, ids, train=False, p_drop_s1=0.3, p_drop_s2=0.1, p_drop_t=0.15, seed=0):
        self.cache, self.ids, self.train = cache, list(ids), train
        self.p_drop_s1, self.p_drop_s2, self.p_drop_t = p_drop_s1, p_drop_s2, p_drop_t
        self.rng = np.random.default_rng(seed)
        self.i_s2 = [PASTIS_DYN.index(b) for b in S2_PASTIS + ['OBS_S2']]
        self.i_s1 = [PASTIS_DYN.index(b) for b in S1_NAMES + ['OBS_S1']]

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        d = np.load(self.cache / f'{self.ids[i]}.npz')
        x = d['x'].astype(np.float32)
        tmask = np.ones(x.shape[0], bool)
        tg = [d['sem'], d['extent'], d['boundary'], d['dist']]
        if self.train:
            rng = self.rng
            x, *tg = augment_spatial([x] + tg, rng)
            r = rng.random()
            if r < self.p_drop_s1: x[:, self.i_s1] = 0                       # SAR missing (as in YieldSAT)
            elif r < self.p_drop_s1 + self.p_drop_s2: x[:, self.i_s2] = 0    # optical missing (clouds)
            drop = rng.random(x.shape[0]) < self.p_drop_t
            if drop.all(): drop[rng.integers(len(drop))] = False
            x[drop] = 0; tmask &= ~drop
        return dict(x=torch.from_numpy(x), days=torch.from_numpy(d['days'].astype(np.float32)), tmask=torch.from_numpy(tmask),
                    sem=torch.from_numpy(tg[0].astype(np.int64)), extent=torch.from_numpy(tg[1].astype(np.float32)),
                    boundary=torch.from_numpy(tg[2].astype(np.float32)), dist=torch.from_numpy(tg[3].astype(np.float32)),
                    id=self.ids[i])

## 3 · Dataset 2: YieldSAT (yield, crop type, fields: Argentina, Brazil, Uruguay, Germany)
[YieldSAT](https://yieldsat.github.io/) (CVPR 2026) provides 10 m combine-harvester yield maps for 2,173 fields, with Sentinel-2 (12 bands), weather, soil and topography.

**Getting the data**: access is granted through the form on the project page. Put the *preprocessed (ML-ready)* release in Drive at
`sar_crop_ml/data/raw/yieldsat/preprocessed-24-ts/<Country>/…/merge_s2-soil-dem-weather-coords.nc`. Any folder depth works; every `*.nc` file below `data/raw/yieldsat/` is picked up.

The NetCDF files are pixel tables (`index × 24 time steps × 120 bands`). Fields are rebuilt as images from `row`/`col`. Bands are split by type:
* **dynamic**: 12 Sentinel-2 bands, plus 4 weather series (field-level, repeated per pixel)
* **static**: soil, topography and coordinates, stored once per pixel instead of 24 times (≈10× smaller cache)

**Targets**: `yield` (t/ha per pixel = "efficiency"), `crop` (crop type), `extent` (pixels belonging to the field).
YieldSAT has **no SAR**. Models trained on PASTIS-R learned to work without it, because the SAR channels are randomly dropped during training.

In [ ]:
S2_YIELDSAT = ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09', 'B11', 'B12', 'B8A']
WEATHER = ['temp_mean', 'temp_max', 'temp_min', 'total_prec']
YS_CROP_PATCH = 64                 # training crop size (fields are padded / cropped to this)
YS_SPLIT_BY = 'field'              # 'field' (paper default), 'farm' or 'year' — leakage-free grouped folds
CACHE_YIELDSAT_ON_DRIVE = True     # the field cache is ~10–15 GB; keeping it on Drive avoids rebuilding it


def make_fake_yieldsat(path, n_fields=14):
    import xarray as xr
    rng = np.random.default_rng(1)
    static = ['aspect', 'clay_0-5', 'dem', 'slope', 'twi', 'coord_x', 'coord_y', 'coord_z']
    bands = S2_YIELDSAT + static + WEATHER
    crops = ['corn', 'rapeseed', 'soybean', 'wheat']
    rows, cols, fid, crop, samples, target, times = [], [], [], [], [], [], []
    for f in range(n_fields):
        h, w = rng.integers(20, 50), rng.integers(20, 50)
        c = f % 4
        rr, cc = np.mgrid[0:h, 0:w]
        keep = rng.random((h, w)) < 0.9
        rr, cc = rr[keep], cc[keep]
        n = len(rr)
        s = np.zeros((n, 24, len(bands)), np.float32)
        s[:, :, :12] = rng.normal(0.2 + 0.05 * c, 0.05, (n, 24, 12))
        s[:, :, 12:12 + len(static)] = rng.normal(10, 2, (n, 1, len(static)))
        s[:, :, 12 + 5:12 + 8] = rng.normal(4e6, 10, (n, 1, 3))           # coordinates: large values
        s[:, :, -4:] = rng.normal(15, 5, (1, 24, 4))
        s[:, :4] = np.nan                                                  # outside growing season
        y = 3 + c + np.sin(rr / 7.0) + 0.1 * rng.normal(size=n) + 2 * s[:, 10, 7]
        rows += [rr]; cols += [cc]; fid += [np.full(n, f)]; crop += [np.full(n, c)]; samples += [s]; target += [y]
        t0 = np.datetime64('2018-09-01') + np.timedelta64(int(rng.integers(0, 30)), 'D')
        times += [np.broadcast_to(t0 + np.arange(24) * np.timedelta64(30, 'D'), (n, 24))]
    N = sum(len(r) for r in rows)
    ds = xr.Dataset(
        dict(sample=(('index', 'time_step', 'band'), np.concatenate(samples)), target=('index', np.concatenate(target).astype(np.float32)),
             row=('index', np.concatenate(rows).astype(np.uint8)), col=('index', np.concatenate(cols).astype(np.uint8)),
             field_shared_name=('index', np.concatenate(fid).astype(np.uint16)), crop=('index', np.concatenate(crop).astype(np.uint8)),
             farm_identifier=('index', (np.concatenate(fid) // 3).astype(np.uint8)), year=('index', np.full(N, 2019, np.uint16)),
             times=(('index', 'time_step'), np.concatenate(times))),
        coords=dict(index=np.arange(N), time_step=np.arange(24), band=bands))
    ds['field_shared_name'].attrs = {str(f): f'Fake_farm{f // 3}_field{f}_{crops[f % 4]}_2019' for f in range(n_fields)}
    ds['crop'].attrs = {str(i): c for i, c in enumerate(crops)}
    path.parent.mkdir(parents=True, exist_ok=True)
    ds.to_netcdf(path)


def _codes(da):
    return {int(k): str(v) for k, v in da.attrs.items() if str(k).lstrip('-').isdigit()}


def prepare_yieldsat(chunk=150_000):
    """Rebuild every field as an image tensor. Resumable: finished fields are skipped."""
    import xarray as xr
    src_dir = P['raw'] / 'yieldsat'
    if SMOKE and not list(src_dir.rglob('*.nc')):
        make_fake_yieldsat(src_dir / 'preprocessed-24-ts/Fakeland/merge_s2-soil-dem-weather-coords.nc')
    files = sorted(src_dir.rglob('*.nc'))
    if not files:
        raise FileNotFoundError(f'No YieldSAT .nc files under {src_dir}. Request access at https://yieldsat.github.io/ and copy the preprocessed release there.')
    out = P['local'] / 'cache' / 'yieldsat'
    drive_out = P['cache'] / 'yieldsat'
    out.mkdir(parents=True, exist_ok=True)
    if CACHE_YIELDSAT_ON_DRIVE and (drive_out / 'index.json').exists() and not (out / 'index.json').exists():
        sh(f'cp -r "{drive_out}/." "{out}/"')
    if (out / 'index.json').exists():
        return out, json.load(open(out / 'index.json'))

    index, stats = [], {}
    for fpath in files:
        parts = fpath.relative_to(src_dir).parts[:-1]
        country = next((p for p in reversed(parts) if p.lower() not in ('merged', 'preprocessed-24-ts', 'yieldsat')), fpath.stem)
        ds = xr.open_dataset(fpath, chunks={'index': chunk}) if _has_dask() else xr.open_dataset(fpath)
        bands = [str(b) for b in ds['band'].values]
        s2 = [b for b in S2_YIELDSAT if b in bands]
        wx = [b for b in WEATHER if b in bands]
        others = [b for b in bands if b not in s2 + wx]
        bi = {b: i for i, b in enumerate(bands)}
        # static = constant over time (checked on the first chunk); anything varying stays dynamic
        probe = ds['sample'].isel(index=slice(0, min(20000, ds.sizes['index']))).values
        def is_static(v):   # constant over time for every pixel (relative tolerance: coordinates are ~1e6)
            with np.errstate(all='ignore'):
                dev = np.nanmax(np.abs(v - np.nanmean(v, 1, keepdims=True)), initial=0)
                return dev <= 1e-5 * (np.nanmax(np.abs(v), initial=0) + 1)
        static = [b for b in others if is_static(probe[:, :, bi[b]])]
        dyn_extra = [b for b in others if b not in static]
        dyn = s2 + wx + dyn_extra
        names = _codes(ds['field_shared_name']); crops = _codes(ds['crop'])
        fid = ds['field_shared_name'].values.astype(np.int64)
        need = np.bincount(fid)
        buf = {}
        print(f'{fpath.name} [{country}]: {ds.sizes["index"]:,} pixels, {len(np.unique(fid))} fields, dyn={len(dyn)} static={len(static)}')
        for s0 in range(0, ds.sizes['index'], chunk):
            sl = slice(s0, min(ds.sizes['index'], s0 + chunk))
            part = ds.isel(index=sl)
            smp = part['sample'].values.astype(np.float32)
            cols = {k: part[k].values for k in ('row', 'col', 'field_shared_name', 'crop', 'target')}
            tim = part['times'].values if 'times' in part else None
            extra = {k: part[k].values for k in ('farm_identifier', 'year') if k in part}
            for f in np.unique(cols['field_shared_name']):
                m = cols['field_shared_name'] == f
                b = buf.setdefault(int(f), dict(rows=[], cols=[], dyn=[], st=[], y=[], crop=int(cols['crop'][m][0]),
                                                 times=None if tim is None else tim[m][0], extra={k: int(v[m][0]) for k, v in extra.items()}))
                b['rows'].append(cols['row'][m]); b['cols'].append(cols['col'][m])
                b['dyn'].append(smp[m][:, :, [bi[x] for x in dyn]]); b['st'].append(smp[m][:, 0, [bi[x] for x in static]])
                b['y'].append(cols['target'][m])
                if sum(len(r) for r in b['rows']) >= need[f]:
                    index.append(_write_field(out, country, names.get(int(f), f'{country}_field{f}'), crops.get(b['crop'], str(b['crop'])),
                                              buf.pop(int(f)), dyn, static, stats))
            print(f'  {sl.stop:,}/{ds.sizes["index"]:,} pixels', flush=True)
        for f, b in buf.items():   # fields whose pixel count was smaller than expected
            index.append(_write_field(out, country, names.get(f, f'{country}_field{f}'), crops.get(b['crop'], str(b['crop'])), b, dyn, static, stats))
    norm = {k: [v[1] / max(v[0], 1), math.sqrt(max(v[2] / max(v[0], 1) - (v[1] / max(v[0], 1)) ** 2, 1e-12))] for k, v in stats.items()}
    meta = dict(fields=index, norm=norm, dyn=dyn, static=static, crops=sorted({f['crop'] for f in index}))
    json.dump(meta, open(out / 'index.json', 'w'))
    if CACHE_YIELDSAT_ON_DRIVE:
        drive_out.mkdir(parents=True, exist_ok=True)
        sh(f'cp -r "{out}/." "{drive_out}/"', check=False)
    return out, meta


def _has_dask():
    try:
        import dask  # noqa
        return True
    except Exception:
        return False


def _acc(stats, name, v):
    v = v[np.isfinite(v)].astype(np.float64)
    s = stats.setdefault(name, [0, 0.0, 0.0])
    s[0] += v.size; s[1] += v.sum(); s[2] += (v ** 2).sum()


def _write_field(out, country, name, crop, b, dyn, static, stats):
    rows, cols = np.concatenate(b['rows']).astype(int), np.concatenate(b['cols']).astype(int)
    D, S, y = np.concatenate(b['dyn']), np.concatenate(b['st']), np.concatenate(b['y']).astype(np.float32)
    H, W = rows.max() + 1, cols.max() + 1
    T = D.shape[1]
    x = np.full((T, len(dyn), H, W), np.nan, np.float32); x[:, :, rows, cols] = D.transpose(1, 2, 0)
    st = np.full((len(static), H, W), np.nan, np.float32); st[:, rows, cols] = S.T
    ym = np.full((H, W), np.nan, np.float32); ym[rows, cols] = y
    mask = np.zeros((H, W), np.uint8); mask[rows, cols] = 1
    for i, n in enumerate(dyn): _acc(stats, n, D[:, :, i])
    for i, n in enumerate(static): _acc(stats, n, S[:, i])
    _acc(stats, '__yield__', y)
    if b['times'] is not None and not np.isnat(b['times'][0]):
        t = b['times']; t0 = t[~np.isnat(t)][0]
        days = np.array([(ti - t0) / np.timedelta64(1, 'D') if not np.isnat(ti) else i * 30 for i, ti in enumerate(t)], np.float32)
    else:
        days = np.arange(T, dtype=np.float32) * 30
    safe = re.sub(r'[^A-Za-z0-9_.-]', '_', name)
    (out / country).mkdir(exist_ok=True)
    np.savez(out / country / f'{safe}.npz', x=x.astype(np.float32 if np.nanmax(np.abs(D), initial=0) > 6e4 else np.float16),
             static=st, y=ym, mask=mask, days=days)
    rec = dict(file=f'{country}/{safe}.npz', name=name, country=country, crop=crop, H=int(H), W=int(W), n=int(mask.sum()),
               mean_yield=float(np.nanmean(y)), **b['extra'])
    rec['farm'] = f"{country}_{rec.get('farm_identifier', name.split('_field')[0])}"
    return rec


def yieldsat_folds(meta, k=5):
    """Grouped folds (no field/farm/year appears in two folds), saved once and shared by all notebooks."""
    sp = P['splits'] / f'yieldsat_{YS_SPLIT_BY}_k{k}.json'
    if sp.exists():
        return json.load(open(sp))
    key = {'field': lambda f: f['name'], 'farm': lambda f: f['farm'], 'year': lambda f: str(f.get('year', '0'))}[YS_SPLIT_BY]
    groups = sorted({key(f) for f in meta['fields']})
    rng = random.Random(42); rng.shuffle(groups)
    gfold = {g: i % k + 1 for i, g in enumerate(groups)}
    folds = {f['name']: gfold[key(f)] for f in meta['fields']}
    json.dump(folds, open(sp, 'w'))
    return folds


def yieldsat_split(meta, folds, test_fold):
    val_fold = test_fold % 5 + 1
    tr = [f for f in meta['fields'] if folds[f['name']] not in (test_fold, val_fold)]
    va = [f for f in meta['fields'] if folds[f['name']] == val_fold]
    te = [f for f in meta['fields'] if folds[f['name']] == test_fold]
    return tr, va, te


class YieldDS(torch.utils.data.Dataset):
    """train: random crops (YS_CROP_PATCH²) · eval: whole field padded to a multiple of 16."""
    def __init__(self, cache, meta, fields, train=False, crops_per_field=6, p_drop_t=0.1, seed=0):
        self.cache, self.meta, self.fields, self.train = cache, meta, fields, train
        self.cpf, self.p_drop_t = crops_per_field, p_drop_t
        self.rng = np.random.default_rng(seed)
        n = meta['norm']
        self.dm = np.array([n[b][0] for b in meta['dyn']], np.float32)[None, :, None, None]
        self.ds = np.array([n[b][1] for b in meta['dyn']], np.float32)[None, :, None, None] + 1e-6
        self.sm = np.array([n[b][0] for b in meta['static']], np.float32)[:, None, None]
        self.ss = np.array([n[b][1] for b in meta['static']], np.float32)[:, None, None] + 1e-6
        self.ym, self.ys = n['__yield__']
        self.crop_idx = {c: i for i, c in enumerate(meta['crops'])}
        self.i_s2 = [i for i, b in enumerate(meta['dyn']) if b in S2_YIELDSAT]

    def __len__(self): return len(self.fields) * (self.cpf if self.train else 1)

    def __getitem__(self, i):
        f = self.fields[i % len(self.fields)]
        d = np.load(self.cache / f['file'])
        x, st, y, mask, days = d['x'].astype(np.float32), d['static'], d['y'], d['mask'], d['days']
        obs = np.isfinite(x[:, self.i_s2]).any(1, keepdims=True).astype(np.float32)        # (T,1,H,W)
        x = np.nan_to_num((x - self.dm) / self.ds)
        st = np.nan_to_num((st - self.sm) / self.ss)
        x = np.concatenate([x, obs], 1)                                                        # + OBS_S2
        tmask = obs.reshape(obs.shape[0], -1).any(1)
        H, W = mask.shape
        s = YS_CROP_PATCH
        if self.train:
            ph, pw = max(0, s - H), max(0, s - W)
            pad = lambda a, v=0: np.pad(a, [(0, 0)] * (a.ndim - 2) + [(0, ph), (0, pw)], constant_values=v)
            x, st, y, mask = pad(x), pad(st), pad(y, np.nan), pad(mask)
            for _ in range(10):   # prefer crops that contain field pixels
                y0, x0 = self.rng.integers(0, x.shape[-2] - s + 1), self.rng.integers(0, x.shape[-1] - s + 1)
                if mask[y0:y0 + s, x0:x0 + s].mean() > 0.2: break
            x, st, y, mask = (a[..., y0:y0 + s, x0:x0 + s] for a in (x, st, y, mask))
            x, st, y, mask = augment_spatial([x, st, y, mask], self.rng)
            drop = (self.rng.random(len(tmask)) < self.p_drop_t) & tmask
            if drop.sum() < tmask.sum(): x[drop] = 0; tmask = tmask & ~drop
        else:
            Hp, Wp = (H + 15) // 16 * 16, (W + 15) // 16 * 16
            pad = lambda a, v=0: np.pad(a, [(0, 0)] * (a.ndim - 2) + [(0, Hp - H), (0, Wp - W)], constant_values=v)
            x, st, y, mask = pad(x), pad(st), pad(y, np.nan), pad(mask)
        valid_y = np.isfinite(y) & (mask > 0)
        crop = np.where(mask > 0, self.crop_idx[f['crop']], 255).astype(np.int64)
        return dict(x=torch.from_numpy(np.ascontiguousarray(x, np.float32)), static=torch.from_numpy(np.ascontiguousarray(st, np.float32)),
                    days=torch.from_numpy(days.astype(np.float32)), tmask=torch.from_numpy(tmask),
                    crop=torch.from_numpy(crop), extent=torch.from_numpy(mask.astype(np.float32)),
                    yld=torch.from_numpy(np.where(valid_y, (np.nan_to_num(y) - self.ym) / self.ys, 0).astype(np.float32)),
                    ymask=torch.from_numpy(valid_y), id=f['name'])


def yieldsat_dyn_bands(meta):
    return list(meta['dyn']) + ['OBS_S2']

## 4 · Shared model parts and weight transfer
Every model starts with a **band-aware input layer**: a per-pixel linear map whose input columns are tied to band *names* (`B04`, `S1A_VV`, `temp_mean`, …), not positions. That makes transfer between datasets mechanical:
* columns for bands present in both datasets are copied (the 10 shared Sentinel-2 bands);
* new bands (`B01`, `B09`, weather) start at zero, so the transferred model initially behaves exactly as before and learns to use them;
* bands that disappear (SAR in YieldSAT) are dropped.

Static layers (soil, terrain) enter through a separate zero-initialised branch. Output heads are named, and the PASTIS crop head initialises the YieldSAT crop head (Corn → corn, Winter rapeseed → rapeseed, Soybeans → soybean, Soft winter wheat → wheat).

In [ ]:
def day_encoding(days, d, period=1000.0):
    """Sinusoidal encoding of acquisition day (days since season start). days (B,T) -> (B,T,d)."""
    i = torch.arange(d // 2, device=days.device, dtype=torch.float32)
    freq = period ** (-2 * i / d)
    a = days.float()[..., None] * freq
    return torch.cat([torch.sin(a), torch.cos(a)], -1)


class BandLinear(nn.Module):
    """Per-pixel linear layer over named input bands: (…, C_in) -> (…, C_out)."""
    def __init__(self, bands, out_dim):
        super().__init__()
        self.bands = list(bands)
        self.lin = nn.Linear(len(self.bands), out_dim)

    def forward(self, x):
        return self.lin(x)

    @torch.no_grad()
    def adapted(self, new_bands):
        new = BandLinear(new_bands, self.lin.out_features).to(self.lin.weight.device)
        new.lin.weight.zero_(); new.lin.bias.copy_(self.lin.bias)
        copied = []
        for j, b in enumerate(new_bands):
            if b in self.bands:
                new.lin.weight[:, j] = self.lin.weight[:, self.bands.index(b)]; copied.append(b)
        print(f'  input layer: copied {len(copied)}/{len(new_bands)} bands {copied}; new (zero-init): {[b for b in new_bands if b not in copied]}')
        return new


class StaticBranch(nn.Module):
    """Static per-pixel layers (soil, DEM, coordinates) -> feature map; output layer zero-initialised."""
    def __init__(self, bands, dim):
        super().__init__()
        self.bands = list(bands)
        self.net = nn.Sequential(nn.Conv2d(len(bands), dim, 1), nn.GELU(), nn.Conv2d(dim, dim, 1))
        nn.init.zeros_(self.net[2].weight); nn.init.zeros_(self.net[2].bias)

    def forward(self, s):
        return self.net(s)


class Heads(nn.Module):
    """Named task heads on a shared feature map."""
    def __init__(self, c, tasks):
        super().__init__()
        self.tasks = dict(tasks)
        self.trunk = nn.Sequential(nn.Conv2d(c, c, 3, padding=1), nn.GroupNorm(min(8, c), c), nn.GELU())
        self.out = nn.ModuleDict({k: nn.Conv2d(c, n, 1) for k, n in self.tasks.items()})

    def forward(self, f):
        f = self.trunk(f)
        return {k: m(f) for k, m in self.out.items()}

    @torch.no_grad()
    def adapted(self, new_tasks, class_map=None):
        """class_map: {'crop': {new_class_index: old_class_index}} copies classifier rows from 'sem'."""
        c = self.trunk[0].in_channels
        new = Heads(c, new_tasks).to(self.trunk[0].weight.device)
        new.trunk.load_state_dict(self.trunk.state_dict())
        for k, n in new_tasks.items():
            if k in self.out and self.out[k].out_channels == n:
                new.out[k].load_state_dict(self.out[k].state_dict()); print(f'  head {k}: transferred')
            elif class_map and k in class_map and 'sem' in self.out:
                for j, i in class_map[k].items():
                    new.out[k].weight[j] = self.out['sem'].weight[i]; new.out[k].bias[j] = self.out['sem'].bias[i]
                print(f'  head {k}: rows initialised from sem classes {class_map[k]}')
            else:
                print(f'  head {k}: new')
        return new


PASTIS_TO_CROP = {'corn': 3, 'maize': 3, 'rapeseed': 5, 'canola': 5, 'soy': 15, 'wheat': 2, 'barley': 4, 'sunflower': 7, 'sorghum': 18, 'potato': 13, 'beet': 9}

def crop_class_map(crop_names):
    m = {}
    for j, c in enumerate(crop_names):
        for k, i in PASTIS_TO_CROP.items():
            if k in c.lower(): m[j] = i; break
    return m


class TemporalPool(nn.Module):
    """Attention pooling over time with a validity mask. x (B,C,T,H,W), mask (B,T) -> (B,C,H,W)."""
    def __init__(self, c):
        super().__init__()
        self.score = nn.Conv3d(c, 1, 1)

    def forward(self, x, mask):
        s = self.score(x).float().masked_fill(~mask[:, None, :, None, None], -1e4)
        w = torch.softmax(s, dim=2).to(x.dtype)
        return (x * w).sum(2)


def count_params(m):
    return sum(p.numel() for p in m.parameters())

## 5 · Model: U-Mamba for satellite image time series
**U-Mamba** ([Ma, Li & Wang 2024, arXiv:2401.04722](https://arxiv.org/abs/2401.04722)) is a U-Net whose encoder stages combine residual convolution blocks with a **Mamba layer that scans all voxels of the stage as one long sequence**, to capture long-range dependencies (the "U-Mamba_Enc" variant). It was designed for 3-D medical volumes.

Changes made for satellite time series:
* **Time is the third axis**: the input is a (time × height × width) volume, so 3-D residual blocks mix space and time. Downsampling is spatial only, so every date is kept.
* Mamba runs on the flattened (T·H·W) tokens of the deeper stages (`MAMBA_STAGES`); the shallow stages would be 400 k tokens long.
* **Acquisition dates** are added as a sinusoidal encoding, because satellite dates are irregular.
* Skip connections are collapsed over time with masked **attention pooling** (missing dates are ignored), and decoding is 2-D.
* Band-aware input layer, a static-data branch and named multi-task heads (section 4).

Note: arXiv 2604.00940, the link in the request, is the YieldSAT dataset paper. The U-Mamba architecture paper is 2401.04722.

In [ ]:
MODEL_NAME = 'umamba'


class Res3d(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        s = (1, stride, stride)
        self.c1 = nn.Conv3d(cin, cout, 3, stride=s, padding=1, bias=False); self.n1 = nn.InstanceNorm3d(cout, affine=True)
        self.c2 = nn.Conv3d(cout, cout, 3, padding=1, bias=False); self.n2 = nn.InstanceNorm3d(cout, affine=True)
        self.skip = nn.Identity() if (cin == cout and stride == 1) else nn.Sequential(nn.Conv3d(cin, cout, 1, stride=s, bias=False), nn.InstanceNorm3d(cout, affine=True))
        self.act = nn.LeakyReLU(0.01, inplace=True)

    def forward(self, x):
        return self.act(self.n2(self.c2(self.act(self.n1(self.c1(x))))) + self.skip(x))


class Res2d(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, padding=1, bias=False); self.n1 = nn.InstanceNorm2d(cout, affine=True)
        self.c2 = nn.Conv2d(cout, cout, 3, padding=1, bias=False); self.n2 = nn.InstanceNorm2d(cout, affine=True)
        self.skip = nn.Identity() if cin == cout else nn.Conv2d(cin, cout, 1, bias=False)
        self.act = nn.LeakyReLU(0.01, inplace=True)

    def forward(self, x):
        return self.act(self.n2(self.c2(self.act(self.n1(self.c1(x))))) + self.skip(x))


class MambaLayer3d(nn.Module):
    """U-Mamba's Mamba layer: LayerNorm + Mamba over all (T·H·W) tokens (residual added for stability)."""
    def __init__(self, c, d_state=16):
        super().__init__()
        self.norm = nn.LayerNorm(c)
        self.mamba = make_mamba(c, d_state=d_state)

    def forward(self, x):
        b, c, t, h, w = x.shape
        tok = x.flatten(2).transpose(1, 2)                      # (B, T·H·W, C)
        out = self.mamba(self.norm(tok))
        return x + out.transpose(1, 2).reshape(b, c, t, h, w)


class UMambaSITS(nn.Module):
    def __init__(self, dyn_bands, static_bands, tasks, widths=(32, 64, 128, 256), blocks=2, mamba_stages=(2, 3), d_state=16):
        super().__init__()
        self.cfg = dict(widths=list(widths), blocks=blocks, mamba_stages=list(mamba_stages), d_state=d_state)
        w0 = widths[0]
        self.stem = BandLinear(dyn_bands, w0)
        self.day = nn.Linear(w0, w0)
        self.enc = nn.ModuleList(); self.mamba = nn.ModuleList(); self.pool = nn.ModuleList()
        cin = w0
        for i, w in enumerate(widths):
            layers = [Res3d(cin, w, stride=2 if i > 0 else 1)] + [Res3d(w, w) for _ in range(blocks - 1)]
            self.enc.append(nn.Sequential(*layers))
            self.mamba.append(MambaLayer3d(w, d_state) if i in mamba_stages else nn.Identity())
            self.pool.append(TemporalPool(w))
            cin = w
        self.up = nn.ModuleList([nn.ConvTranspose2d(widths[i + 1], widths[i], 2, stride=2) for i in range(len(widths) - 1)])
        self.dec = nn.ModuleList([Res2d(2 * widths[i], widths[i]) for i in range(len(widths) - 1)])
        self.static = StaticBranch(static_bands, w0) if static_bands else None
        self.static_deep = self._zero_conv(w0, widths[-1]) if static_bands else None
        self.heads = Heads(w0, tasks)

    @staticmethod
    def _zero_conv(cin, cout):
        c = nn.Conv2d(cin, cout, 1); nn.init.zeros_(c.weight); nn.init.zeros_(c.bias); return c

    def forward(self, batch):
        x, days, tmask = batch['x'], batch['days'], batch['tmask']           # x (B,T,C,H,W)
        b, t, c, h, w = x.shape
        z = self.stem(x.permute(0, 1, 3, 4, 2))                             # (B,T,H,W,E)
        z = z + self.day(day_encoding(days, z.shape[-1]))[:, :, None, None, :]
        z = z.permute(0, 4, 1, 2, 3).contiguous()                           # (B,E,T,H,W)
        skips = []
        for enc, mam, pool in zip(self.enc, self.mamba, self.pool):
            z = mam(enc(z))
            skips.append(pool(z, tmask))
        f = skips[-1]
        s = None
        if self.static is not None and batch.get('static') is not None:
            s = self.static(batch['static'])
            f = f + self.static_deep(F.adaptive_avg_pool2d(s, f.shape[-2:]))
        for i in reversed(range(len(self.up))):
            f = self.dec[i](torch.cat([self.up[i](f), skips[i]], 1))
        if s is not None:
            f = f + s
        return self.heads(f)

    def adapt(self, dyn_bands, static_bands, tasks, class_map=None):
        """Transfer to a new dataset: remap input bands, add a static branch, re-head."""
        w0 = self.cfg['widths'][0]
        self.stem = self.stem.adapted(dyn_bands)
        if static_bands:
            self.static = StaticBranch(static_bands, w0).to(self.day.weight.device)
            self.static_deep = self._zero_conv(w0, self.cfg['widths'][-1]).to(self.day.weight.device)
            print(f'  static branch: new, {len(static_bands)} bands (zero-initialised output)')
        else:
            self.static = self.static_deep = None
        self.heads = self.heads.adapted(tasks, class_map)
        return self

    def backbone_parameters(self):
        skip = set(map(id, list(self.stem.parameters()) + list(self.heads.parameters()) +
                       (list(self.static.parameters()) + list(self.static_deep.parameters()) if self.static is not None else [])))
        return [p for p in self.parameters() if id(p) not in skip]


def build_model(dyn_bands, static_bands, tasks, smoke=SMOKE):
    if smoke:
        return UMambaSITS(dyn_bands, static_bands, tasks, widths=(8, 16, 32), blocks=1, mamba_stages=(1, 2), d_state=4)
    return UMambaSITS(dyn_bands, static_bands, tasks, widths=(32, 64, 128, 256), blocks=2, mamba_stages=(2, 3), d_state=16)

def MODEL_EXTRA_LOSSES_FN(dyn_bands):
    return {}   # U-Mamba is trained with the task losses only

DEFAULT_EPOCHS = {'pastis': 100, 'yieldsat': (10, 60)}
DEFAULT_BATCH = {'pastis': 4 if GPU_MEM_GB < 60 else 8, 'yieldsat': 16 if GPU_MEM_GB < 60 else 32}
DEFAULT_LR = {'pastis': 1e-3, 'yieldsat': 1e-3}

## 6 · Training engine: losses, metrics, crash-safe checkpoints
* `last.pt` is written to Drive every `CKPT_MINUTES` and at the end of every epoch. It holds the model, optimizer, scheduler, AMP scaler, RNG states, epoch, **position inside the epoch** and the best score, so a run that dies mid-epoch continues from the same batch.
* `best.pt` is the best validation checkpoint; `metrics.csv` gets one row per epoch; `log.txt` keeps the console log.
* To stop a run cleanly from another notebook or from Drive, create an empty file called `STOP` in the run folder.
* Writes are atomic (temporary file, then rename), so a disconnect during a save cannot corrupt the previous checkpoint.

In [ ]:
import csv, inspect, io, contextlib
CKPT_MINUTES = 10
NUM_WORKERS = 0 if SMOKE else min(8, os.cpu_count() or 2)

PASTIS_TASKS = {'sem': len(PASTIS_CLASSES), 'extent': 1, 'boundary': 1, 'dist': 1}
PASTIS_LOSS_W = {'sem': 1.0, 'extent': 0.5, 'boundary': 0.5, 'dist': 0.5}
YIELD_LOSS_W = {'yld': 1.0, 'crop': 0.5, 'extent': 0.25}


def task_losses(out, batch, weights):
    L = {}
    if 'sem' in weights and 'sem' in out:
        L['sem'] = F.cross_entropy(out['sem'].float(), batch['sem'], ignore_index=255)
    if 'crop' in weights and 'crop' in out:
        L['crop'] = F.cross_entropy(out['crop'].float(), batch['crop'], ignore_index=255)
    if 'extent' in weights and 'extent' in out:
        L['extent'] = F.binary_cross_entropy_with_logits(out['extent'][:, 0].float(), batch['extent'])
    if 'boundary' in weights and 'boundary' in out:
        L['boundary'] = F.binary_cross_entropy_with_logits(out['boundary'][:, 0].float(), batch['boundary'],
                                                           pos_weight=torch.tensor(5.0, device=batch['boundary'].device))
    if 'dist' in weights and 'dist' in out:
        m = batch['extent']
        L['dist'] = (torch.abs(torch.sigmoid(out['dist'][:, 0].float()) - batch['dist']) * m).sum() / (m.sum() + 1e-6)
    if 'yld' in weights and 'yld' in out:
        m = batch['ymask'].float()
        L['yld'] = (F.huber_loss(out['yld'][:, 0].float(), batch['yld'], reduction='none', delta=1.0) * m).sum() / (m.sum() + 1e-6)
    total = sum(weights[k] * v for k, v in L.items())
    return total, L


class Metrics:
    """Streaming metrics for every task present in the batch."""
    def __init__(self, n_sem=None, n_crop=None, yield_norm=None, ignore_sem=()):
        self.cm = np.zeros((n_sem, n_sem), np.int64) if n_sem else None
        self.cc = np.zeros((n_crop, n_crop), np.int64) if n_crop else None
        self.ignore_sem = set(ignore_sem)
        self.bin = {k: np.zeros(3) for k in ('extent', 'boundary')}   # tp, fp, fn
        self.dist = [0.0, 0]
        self.yn = yield_norm
        self.y = [0.0, 0.0, 0.0, 0.0, 0.0, 0]     # sum err², sum |err|, sum y, sum y², n... for R²
        self.fields = []                          # (true mean, pred mean, crop true, crop pred) per field

    @staticmethod
    def _cm(cm, pred, tgt):
        k = tgt != 255
        n = cm.shape[0]
        cm += np.bincount(n * tgt[k] + pred[k], minlength=n * n).reshape(n, n)

    def update(self, out, batch):
        if self.cm is not None and 'sem' in out:
            self._cm(self.cm, out['sem'].argmax(1).cpu().numpy().ravel(), batch['sem'].cpu().numpy().ravel())
        for k in self.bin:
            if k in out and k in batch:
                p = (out[k][:, 0] > 0).cpu().numpy(); t = batch[k].cpu().numpy() > 0.5
                self.bin[k] += [(p & t).sum(), (p & ~t).sum(), (~p & t).sum()]
        if 'dist' in out and 'dist' in batch:
            m = batch['extent'].cpu().numpy() > 0.5
            self.dist[0] += np.abs(torch.sigmoid(out['dist'][:, 0].float()).cpu().numpy() - batch['dist'].cpu().numpy())[m].sum(); self.dist[1] += m.sum()
        if 'yld' in out and 'ymask' in batch:
            ym, ys = self.yn
            pr = out['yld'][:, 0].float().cpu().numpy() * ys + ym
            tr = batch['yld'].cpu().numpy() * ys + ym
            msk = batch['ymask'].cpu().numpy()
            e = (pr - tr)[msk]
            self.y[0] += (e ** 2).sum(); self.y[1] += np.abs(e).sum(); self.y[2] += tr[msk].sum(); self.y[3] += (tr[msk] ** 2).sum(); self.y[5] += msk.sum()
            cp = out['crop'].argmax(1).cpu().numpy() if 'crop' in out else None
            for b in range(pr.shape[0]):
                mb = msk[b]
                if mb.sum() == 0: continue
                ct = int(batch['crop'][b].cpu().numpy()[mb][0]) if 'crop' in batch else -1
                cpb = int(np.bincount(cp[b][mb]).argmax()) if cp is not None else -1
                self.fields.append((float(tr[b][mb].mean()), float(pr[b][mb].mean()), ct, cpb))
        if self.cc is not None and 'crop' in out:
            self._cm(self.cc, out['crop'].argmax(1).cpu().numpy().ravel(), batch['crop'].cpu().numpy().ravel())

    @staticmethod
    def _iou(cm, ignore=()):
        inter = np.diag(cm).astype(float); union = cm.sum(0) + cm.sum(1) - np.diag(cm)
        iou = np.where(union > 0, inter / np.maximum(union, 1), np.nan)
        keep = [i for i in range(len(iou)) if i not in ignore]
        return iou, float(np.nanmean(iou[keep])), float(inter.sum() / max(cm.sum(), 1))

    def result(self):
        r = {}
        if self.cm is not None and self.cm.sum():
            iou, miou, oa = self._iou(self.cm, self.ignore_sem)
            r.update(mIoU=miou, OA=oa, per_class_IoU=[None if np.isnan(v) else float(v) for v in iou])
        if self.cc is not None and self.cc.sum():
            iou, miou, oa = self._iou(self.cc)
            r.update(crop_pixel_acc=oa, crop_mIoU=miou)
        for k, (tp, fp, fn) in self.bin.items():
            if tp + fp + fn > 0:
                r[f'{k}_IoU'] = float(tp / max(tp + fp + fn, 1)); r[f'{k}_F1'] = float(2 * tp / max(2 * tp + fp + fn, 1))
        if self.dist[1]: r['dist_MAE'] = float(self.dist[0] / self.dist[1])
        if self.y[5]:
            n = self.y[5]; var = self.y[3] / n - (self.y[2] / n) ** 2
            r.update(pixel_RMSE=float(math.sqrt(self.y[0] / n)), pixel_MAE=float(self.y[1] / n), pixel_R2=float(1 - (self.y[0] / n) / max(var, 1e-9)))
        if self.fields:
            a = np.array([f[:2] for f in self.fields]); t, p = a[:, 0], a[:, 1]
            r.update(field_RMSE=float(np.sqrt(((p - t) ** 2).mean())), field_MAE=float(np.abs(p - t).mean()),
                     field_R2=float(1 - ((p - t) ** 2).sum() / max(((t - t.mean()) ** 2).sum(), 1e-9)), n_fields=len(self.fields))
            c = np.array([f[2:] for f in self.fields])
            if (c[:, 0] >= 0).any(): r['crop_field_acc'] = float((c[:, 0] == c[:, 1]).mean())
        return r


def to_device(batch):
    return {k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}


def amp_ctx():
    if DEVICE != 'cuda': return contextlib.nullcontext()
    return torch.autocast('cuda', dtype=torch.bfloat16 if BF16 else torch.float16)


class EpochSampler(torch.utils.data.Sampler):
    """Deterministic shuffle per (seed, epoch) that can start in the middle: makes mid-epoch resume exact."""
    def __init__(self, n, seed, epoch, start=0):
        self.idx = np.random.default_rng(seed * 1000 + epoch).permutation(n)[start:].tolist()
    def __iter__(self): return iter(self.idx)
    def __len__(self): return len(self.idx)


class Run:
    def __init__(self, stage, name, cfg):
        self.dir = P['runs'] / MODEL_NAME / stage / name
        (self.dir / 'checkpoints').mkdir(parents=True, exist_ok=True)
        (self.dir / 'samples').mkdir(exist_ok=True)
        self.cfg = cfg
        old = self.dir / 'config.json'
        if old.exists():
            prev = json.load(open(old))
            diff = {k: (prev.get(k), v) for k, v in cfg.items() if prev.get(k) != v}
            if diff: self.log(f'config differs from the saved run (saved, now): {diff}')
        json.dump(cfg, open(old, 'w'), indent=1, default=str)
        env = dict(torch=torch.__version__, gpu=GPU, cuda=torch.version.cuda, python=sys.version.split()[0],
                   mamba='cuda' if MambaCUDA else 'pytorch', time=dt.datetime.now(dt.timezone.utc).isoformat())
        json.dump(env, open(self.dir / 'env.json', 'w'), indent=1)

    def log(self, msg):
        line = f'[{dt.datetime.now():%Y-%m-%d %H:%M:%S}] {msg}'
        print(line, flush=True)
        with open(self.dir / 'log.txt', 'a') as f: f.write(line + '\n')

    def save(self, state, name):
        p = self.dir / 'checkpoints' / name
        buf = io.BytesIO(); torch.save(state, buf)
        tmp = p.with_suffix('.tmp'); tmp.write_bytes(buf.getvalue()); os.replace(tmp, p)

    def load(self, name):
        p = self.dir / 'checkpoints' / name
        return torch.load(p, map_location='cpu', weights_only=False) if p.exists() else None

    def append_metrics(self, row):
        p = self.dir / 'metrics.csv'
        rows = list(csv.DictReader(open(p))) if p.exists() else []
        rows = [r for r in rows if int(r['epoch']) < row['epoch']] + [{k: v for k, v in row.items()}]
        keys = list(dict.fromkeys(k for r in rows for k in r))
        with open(p.with_suffix('.tmp'), 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); w.writerows(rows)
        os.replace(p.with_suffix('.tmp'), p)

    def stop_requested(self):
        return (self.dir / 'STOP').exists()


def rng_state():
    return dict(py=random.getstate(), np=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None)

def set_rng_state(s):
    random.setstate(s['py']); np.random.set_state(s['np']); torch.set_rng_state(s['torch'])
    if s.get('cuda') is not None and DEVICE == 'cuda': torch.cuda.set_rng_state_all(s['cuda'])


def make_optimizer(model, phase):
    if phase.get('freeze_backbone'):
        bb = set(map(id, model.backbone_parameters()))
        for p in model.parameters(): p.requires_grad = id(p) not in bb
        groups = [{'params': [p for p in model.parameters() if p.requires_grad], 'lr': phase['lr']}]
    else:
        for p in model.parameters(): p.requires_grad = True
        bb = model.backbone_parameters(); bbi = set(map(id, bb))
        head = [p for p in model.parameters() if id(p) not in bbi]
        groups = [{'params': bb, 'lr': phase['lr'] * phase.get('backbone_lr_mult', 1.0)}, {'params': head, 'lr': phase['lr']}]
    return torch.optim.AdamW(groups, weight_decay=phase.get('wd', 1e-4))


def make_scheduler(opt, total_steps, warmup=0.03):
    w = max(1, int(total_steps * warmup))
    return torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / w) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / max(1, total_steps)))))


def fit(model, run, ds_train, ds_val, phases, batch_size, loss_w, metrics_fn, monitor, mode='max', patience=None,
        extra_losses=None, sample_fn=None, seed=0, val_batch_size=None):
    """phases: list of dicts {epochs, lr, freeze_backbone?, backbone_lr_mult?}. Resumes automatically."""
    extra_losses = extra_losses or {}
    model.to(DEVICE)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda' and not BF16))
    ck = run.load('last.pt')
    st = dict(phase=0, epoch=0, batch=0, step=0, best=None, bad=0, done=False)
    if ck is not None:
        model.load_state_dict(ck['model']); st.update(ck['state'])
        run.log(f"resuming: phase {st['phase']} epoch {st['epoch']} batch {st['batch']} (best {monitor} = {st['best']})")
    if st['done']:
        run.log('run already finished'); return st
    total_epochs = sum(p['epochs'] for p in phases)
    while st['phase'] < len(phases):
        ph = phases[st['phase']]
        opt = make_optimizer(model, ph)
        steps_per_epoch = math.ceil(len(ds_train) / batch_size)
        sched = make_scheduler(opt, ph['epochs'] * steps_per_epoch)
        if ck is not None and ck['state']['phase'] == st['phase']:
            opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched']); scaler.load_state_dict(ck['scaler'])
            if ck.get('rng'): set_rng_state(ck['rng'])
        ck = None
        ep0 = sum(p['epochs'] for p in phases[:st['phase']])
        while st['epoch'] < ep0 + ph['epochs']:
            model.train()
            sampler = EpochSampler(len(ds_train), seed, st['epoch'], start=st['batch'] * batch_size)
            dl = torch.utils.data.DataLoader(ds_train, batch_size=batch_size, sampler=sampler, num_workers=NUM_WORKERS,
                                             pin_memory=DEVICE == 'cuda', drop_last=False, persistent_workers=False)
            t0, last_save, sums, n = time.time(), time.time(), {}, 0
            for batch in dl:
                batch = to_device(batch)
                with amp_ctx():
                    out = model(batch)
                    loss, parts = task_losses(out, batch, loss_w)
                    for k, (w, fn) in extra_losses.items():
                        parts[k] = fn(out, batch); loss = loss + w * parts[k]
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step()
                st['batch'] += 1; st['step'] += 1; n += 1
                for k, v in parts.items(): sums[k] = sums.get(k, 0) + float(v.detach())
                sums['loss'] = sums.get('loss', 0) + float(loss.detach())
                if n % 50 == 0:
                    run.log(f"ep {st['epoch'] + 1}/{total_epochs} it {st['batch']}/{steps_per_epoch} loss {sums['loss'] / n:.4f} lr {sched.get_last_lr()[-1]:.2e}")
                stop = run.stop_requested()
                if time.time() - last_save > CKPT_MINUTES * 60 or stop:
                    run.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(),
                                  state=dict(st), rng=rng_state(), cfg=run.cfg), 'last.pt')
                    last_save = time.time()
                    if stop:
                        run.log('STOP file found: checkpoint saved, stopping'); return st
            # ---- end of epoch: validate
            val = evaluate(model, ds_val, metrics_fn, batch_size=val_batch_size or batch_size)
            score = val.get(monitor)
            improved = score is not None and (st['best'] is None or (score > st['best'] if mode == 'max' else score < st['best']))
            st['epoch'] += 1; st['batch'] = 0
            if improved:
                st['best'] = score; st['bad'] = 0
                run.save(dict(model=model.state_dict(), state=dict(st), metrics=val, cfg=run.cfg), 'best.pt')
            else:
                st['bad'] += 1
            row = dict(epoch=st['epoch'], phase=st['phase'], time_s=round(time.time() - t0, 1), lr=sched.get_last_lr()[-1],
                       **{f'train_{k}': v / max(n, 1) for k, v in sums.items()}, **{f'val_{k}': v for k, v in val.items() if not isinstance(v, list)})
            run.append_metrics(row)
            run.log(f"epoch {st['epoch']}/{total_epochs} · val {monitor} {score:.4f} {'(best)' if improved else ''} · " +
                    ' '.join(f'{k}={v:.4f}' for k, v in val.items() if isinstance(v, float) and k != monitor))
            if sample_fn and (st['epoch'] % 5 == 0 or st['epoch'] == 1 or improved):
                try: sample_fn(model, ds_val, run.dir / 'samples' / f"epoch_{st['epoch']:03d}.png")
                except Exception as e: run.log(f'sample plot failed: {e}')
            stop = patience is not None and st['bad'] >= patience and st['phase'] == len(phases) - 1
            run.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(),
                          state=dict(st), rng=rng_state(), cfg=run.cfg), 'last.pt')
            if stop:
                run.log(f'early stopping ({patience} epochs without improvement)'); st['epoch'] = ep0 + ph['epochs']; break
            if run.stop_requested():
                run.log('STOP file found, stopping'); return st
        st['phase'] += 1
    st['done'] = True
    run.save(dict(model=model.state_dict(), state=dict(st), cfg=run.cfg), 'last.pt')
    run.log(f"finished · best {monitor} = {st['best']}")
    return st


@torch.no_grad()
def evaluate(model, ds, metrics_fn, batch_size=4):
    model.eval()
    m = metrics_fn()
    dl = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    for batch in dl:
        batch = to_device(batch)
        with amp_ctx():
            out = model(batch)
        m.update({k: v.float() for k, v in out.items()}, batch)
    return m.result()


def test_run(model, run, ds_test, metrics_fn, batch_size, sample_fn=None, extra=None):
    """Evaluate best.pt on the test split; writes test_metrics.json (+ a sample figure)."""
    ck = run.load('best.pt') or run.load('last.pt')
    model.load_state_dict(ck['model']); model.to(DEVICE)
    res = evaluate(model, ds_test, metrics_fn, batch_size)
    res.update(params=count_params(model), best_epoch=ck['state']['epoch'], **(extra or {}))
    json.dump(res, open(run.dir / 'test_metrics.json', 'w'), indent=1)
    try: src = inspect.getsource(type(model))
    except Exception: src = '# source unavailable in this kernel\n'
    (run.dir / 'model_architecture.txt').write_text(f'{type(model).__name__} {getattr(model, "cfg", {})}\n\n{model}\n\n{src}')
    if sample_fn: sample_fn(model, ds_test, run.dir / 'samples' / 'test.png')
    run.log('test: ' + ' '.join(f'{k}={v:.4f}' for k, v in res.items() if isinstance(v, float)))
    return res

## 7 · Stage 1: train on PASTIS-R (crop type + field delineation, SAR + optical)
Official 5-fold protocol: test fold `k`, validation fold `k+1`, train on the other three. Set `PASTIS_FOLDS = [1, 2, 3, 4, 5]` for the full cross-validation (each fold is its own resumable run).

Downloading and preprocessing happen once. The 54 GB zip is kept in Drive, and the preprocessed patches stay on the local SSD (or on Drive with `CACHE_PASTIS_ON_DRIVE`).

In [ ]:
PASTIS_FOLDS = [1]
PASTIS_EPOCHS = 2 if SMOKE else DEFAULT_EPOCHS['pastis']
PASTIS_BATCH = 2 if SMOKE else DEFAULT_BATCH['pastis']
PASTIS_LR = DEFAULT_LR['pastis']

pastis_cache, pastis_folds = prepare_pastis()
print(len(pastis_folds), 'patches cached in', pastis_cache)


def rgb_from(x, bands, names=('B04', 'B03', 'B02')):
    idx = [bands.index(n) for n in names]
    t = x.shape[0] // 2
    img = x[t, idx].transpose(1, 2, 0)
    return np.clip((img - np.percentile(img, 2)) / (np.percentile(img, 98) - np.percentile(img, 2) + 1e-6), 0, 1)


@torch.no_grad()
def pastis_samples(model, ds, path, n=4):
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    model.eval()
    n = max(1, min(n, len(ds)))
    cmap = plt.get_cmap('tab20', 20)
    fig, ax = plt.subplots(n, 5, figsize=(15, 3 * n), squeeze=False)
    for r in range(n):
        b = ds[r * len(ds) // n]
        bb = to_device({k: (v[None] if torch.is_tensor(v) else v) for k, v in b.items()})
        with amp_ctx():
            o = {k: v.float() for k, v in model(bb).items()}
        x = b['x'].numpy()
        panels = [(rgb_from(x, PASTIS_DYN), None, 'Sentinel-2 RGB (mid-season)'),
                  (np.where(b['sem'].numpy() == 255, 19, b['sem'].numpy()), cmap, 'crop type: truth'),
                  (o['sem'][0].argmax(0).cpu().numpy(), cmap, 'crop type: predicted'),
                  (torch.sigmoid(o['extent'][0, 0]).cpu().numpy(), 'viridis', 'field extent'),
                  (torch.sigmoid(o['boundary'][0, 0]).cpu().numpy(), 'magma', 'field boundaries')]
        for c, (img, cm, title) in enumerate(panels):
            ax[r, c].imshow(img, cmap=cm, vmin=0 if cm is cmap else None, vmax=19 if cm is cmap else None, interpolation='nearest')
            ax[r, c].set_axis_off()
            if r == 0: ax[r, c].set_title(title, fontsize=10)
    plt.tight_layout(); fig.savefig(path, dpi=90); plt.close(fig)


def fields_from_prediction(out, thr=0.5):
    """Plantation locations as individual fields: watershed on the distance map inside the predicted extent."""
    from skimage.segmentation import watershed
    from skimage.measure import label
    ext = torch.sigmoid(out['extent'][0, 0]).cpu().numpy() > thr
    dist = torch.sigmoid(out['dist'][0, 0]).cpu().numpy()
    markers = label((dist > 0.5) & ext)
    return watershed(-dist, markers, mask=ext)


pastis_results = {}
for fold in PASTIS_FOLDS:
    tr, va, te = pastis_fold_split(pastis_folds, fold)
    cfg = dict(model=MODEL_NAME, stage='pastis', fold=fold, epochs=PASTIS_EPOCHS, batch=PASTIS_BATCH, lr=PASTIS_LR,
               grid=PASTIS_GRID, bands=PASTIS_DYN, tasks=PASTIS_TASKS, loss_w=PASTIS_LOSS_W, n_train=len(tr), n_val=len(va), n_test=len(te))
    run = Run('pastis', f"fold{fold}{globals().get('RUN_SUFFIX', '')}", cfg)
    model = build_model(PASTIS_DYN, [], PASTIS_TASKS)
    run.log(f'{MODEL_NAME}: {count_params(model) / 1e6:.2f} M parameters · train {len(tr)} / val {len(va)} / test {len(te)} patches')
    mfn = lambda: Metrics(n_sem=len(PASTIS_CLASSES), ignore_sem=(PASTIS_VOID,))
    fit(model, run, PastisDS(pastis_cache, tr, train=True, seed=fold), PastisDS(pastis_cache, va),
        phases=[dict(epochs=PASTIS_EPOCHS, lr=PASTIS_LR)], batch_size=PASTIS_BATCH, loss_w=PASTIS_LOSS_W,
        metrics_fn=mfn, monitor='mIoU', mode='max', patience=None if SMOKE else 25,
        extra_losses=MODEL_EXTRA_LOSSES_FN(PASTIS_DYN), sample_fn=pastis_samples, seed=fold)
    pastis_results[fold] = test_run(model, run, PastisDS(pastis_cache, te), mfn, PASTIS_BATCH, sample_fn=pastis_samples)
print(json.dumps({k: {m: v[m] for m in ('mIoU', 'OA', 'extent_IoU', 'boundary_F1') if m in v} for k, v in pastis_results.items()}, indent=1))

## 8 · Stage 2: transfer to YieldSAT (yield = efficiency, crop type, field extent)
The best PASTIS-R checkpoint of this model is loaded and **adapted**:
1. input layer: 10 shared Sentinel-2 bands copied by name; `B01`, `B09`, weather and the observation flag added (zero-initialised); SAR columns dropped;
2. new static branch for soil and terrain (zero-initialised, so it starts neutral);
3. heads: `extent` transferred as is; `crop` initialised from the matching PASTIS crop rows; `yld` (yield regression) is new.

Training then has two phases: **(a)** backbone frozen, so only the new parts learn; **(b)** everything unfrozen, with a lower learning rate on the backbone.
Set `TRANSFER_FROM_PASTIS = False` to train the same architecture from scratch and measure what the transfer is worth.

In [ ]:
TRANSFER_FROM_PASTIS = True
YS_FOLDS = [1]
YS_EPOCHS = (1, 1) if SMOKE else DEFAULT_EPOCHS['yieldsat']      # (frozen, unfrozen) when transferring
YS_BATCH = 4 if SMOKE else DEFAULT_BATCH['yieldsat']
YS_LR = DEFAULT_LR['yieldsat']
SOURCE_PASTIS_FOLD = PASTIS_FOLDS[0] if 'PASTIS_FOLDS' in globals() else 1

ys_cache, ys_meta = prepare_yieldsat()
ys_folds = yieldsat_folds(ys_meta)
YS_DYN, YS_STATIC = yieldsat_dyn_bands(ys_meta), list(ys_meta['static'])
YS_TASKS = {'crop': len(ys_meta['crops']), 'extent': 1, 'yld': 1}
print(f"{len(ys_meta['fields'])} fields · crops {ys_meta['crops']} · {len(YS_DYN)} dynamic + {len(YS_STATIC)} static bands")


@torch.no_grad()
def yield_samples(model, ds, path, n=4):
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    model.eval()
    n = max(1, min(n, len(ds)))
    fig, ax = plt.subplots(n, 4, figsize=(13, 3.2 * n), squeeze=False)
    ym, ys = ds.ym, ds.ys
    for r in range(n):
        b = ds[r * len(ds) // n]
        bb = to_device({k: (v[None] if torch.is_tensor(v) else v) for k, v in b.items()})
        with amp_ctx():
            o = {k: v.float() for k, v in model(bb).items()}
        msk = b['ymask'].numpy()
        tru = np.where(msk, b['yld'].numpy() * ys + ym, np.nan)
        pre = np.where(b['extent'].numpy() > 0, o['yld'][0, 0].cpu().numpy() * ys + ym, np.nan)
        vmin, vmax = np.nanpercentile(tru, 2), np.nanpercentile(tru, 98)
        x = b['x'].numpy()
        panels = [(rgb_from(x, YS_DYN), None, 'Sentinel-2 RGB'), (tru, 'RdYlGn', 'yield t/ha: truth'),
                  (pre, 'RdYlGn', 'yield t/ha: predicted'), (o['crop'][0].argmax(0).cpu().numpy() * (b['extent'].numpy() > 0), 'tab10', 'crop type: predicted')]
        for c, (img, cm, title) in enumerate(panels):
            im = ax[r, c].imshow(img, cmap=cm, vmin=vmin if 'yield' in title else None, vmax=vmax if 'yield' in title else None, interpolation='nearest')
            ax[r, c].set_axis_off()
            if 'yield' in title: plt.colorbar(im, ax=ax[r, c], fraction=0.046)
            if r == 0: ax[r, c].set_title(title, fontsize=10)
        ax[r, 0].text(2, 6, f"{b['id'][:40]} · {ys_meta['crops'][int(b['crop'][msk][0])] if msk.any() else ''}", color='w', fontsize=7)
    plt.tight_layout(); fig.savefig(path, dpi=90); plt.close(fig)


ys_results = {}
for fold in YS_FOLDS:
    tr, va, te = yieldsat_split(ys_meta, ys_folds, fold)
    name = f"{'transfer' if TRANSFER_FROM_PASTIS else 'scratch'}_fold{fold}{globals().get('RUN_SUFFIX', '')}"
    if TRANSFER_FROM_PASTIS:
        phases = [dict(epochs=YS_EPOCHS[0], lr=YS_LR, freeze_backbone=True), dict(epochs=YS_EPOCHS[1], lr=YS_LR * 0.3, backbone_lr_mult=0.3)]
    else:
        phases = [dict(epochs=sum(YS_EPOCHS), lr=YS_LR)]
    cfg = dict(model=MODEL_NAME, stage='yieldsat', fold=fold, transfer=TRANSFER_FROM_PASTIS, source_pastis_fold=SOURCE_PASTIS_FOLD,
               phases=phases, batch=YS_BATCH, crop_size=YS_CROP_PATCH, split_by=YS_SPLIT_BY, dyn=YS_DYN, static=YS_STATIC,
               tasks=YS_TASKS, crops=ys_meta['crops'], loss_w=YIELD_LOSS_W, n_train=len(tr), n_val=len(va), n_test=len(te))
    run = Run('yieldsat', name, cfg)
    if TRANSFER_FROM_PASTIS:
        model = build_model(PASTIS_DYN, [], PASTIS_TASKS)
        src = P['runs'] / MODEL_NAME / 'pastis' / f"fold{SOURCE_PASTIS_FOLD}{globals().get('RUN_SUFFIX', '')}" / 'checkpoints' / 'best.pt'
        if (run.dir / 'checkpoints' / 'last.pt').exists():
            run.log('resuming: architecture adapted without reloading PASTIS weights (they are inside last.pt)')
        else:
            if not src.exists():
                raise FileNotFoundError(f'{src} not found: run stage 1 first, or set TRANSFER_FROM_PASTIS = False')
            model.load_state_dict(torch.load(src, map_location='cpu', weights_only=False)['model'])
            run.log(f'loaded PASTIS weights from {src}')
        model.adapt(YS_DYN, YS_STATIC, YS_TASKS, class_map={'crop': crop_class_map(ys_meta['crops'])})
    else:
        model = build_model(YS_DYN, YS_STATIC, YS_TASKS)
    model.to(DEVICE)
    run.log(f'{MODEL_NAME}: {count_params(model) / 1e6:.2f} M parameters · train {len(tr)} / val {len(va)} / test {len(te)} fields')
    norm = tuple(ys_meta['norm']['__yield__'])
    mfn = lambda: Metrics(n_crop=len(ys_meta['crops']), yield_norm=norm)
    fit(model, run, YieldDS(ys_cache, ys_meta, tr, train=True, seed=fold), YieldDS(ys_cache, ys_meta, va),
        phases=phases, batch_size=YS_BATCH, val_batch_size=1, loss_w=YIELD_LOSS_W, metrics_fn=mfn,
        monitor='pixel_RMSE', mode='min', patience=None if SMOKE else 15,
        extra_losses=MODEL_EXTRA_LOSSES_FN(YS_DYN), sample_fn=yield_samples, seed=fold)
    ys_results[fold] = test_run(model, run, YieldDS(ys_cache, ys_meta, te), mfn, 1, sample_fn=yield_samples)
print(json.dumps({k: {m: v.get(m) for m in ('pixel_RMSE', 'pixel_R2', 'field_RMSE', 'field_R2', 'crop_field_acc', 'extent_IoU')} for k, v in ys_results.items()}, indent=1))

## 9 · Where everything is saved

In [ ]:
for stage in ('pastis', 'yieldsat'):
    for d in sorted((P['runs'] / MODEL_NAME / stage).glob('*')):
        files = sorted(p.relative_to(d).as_posix() for p in d.rglob('*') if p.is_file())
        print(f'{d}:'); [print('   ', f) for f in files[:20]]